## 디지털 사회에서 프라이버시는 계층적으로 거래되는가?

### 가설



### 변수 

- 독립변수: 
- 원 종속변수: `GIVEPIFR`, 1=매우 동의, 5=매우 반대
- 분석 종속변수: `GIVEPIFR_R = 6 - GIVEPIFR`, 1=매우 반대, 5=매우 동의


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 1. 데이터 불러오기

In [ ]:
DATA_PATH = Path(r"/home/pck/data_jour/sda_Team Project_/adt_2025_drop_0509.csv")
df_raw = pd.read_csv(DATA_PATH)

print(df_raw.shape)
display(df_raw.head())

## 2. 분석 변수 지정

분석에 사용할 핵심 변수를 지정한다. 

In [ ]:
raw_dv = "GIVEPIFR"
dv = "GIVEPIFR_R"

raw_iv = ["INCOM0", "RANK", "SATFIN", "FINPROS"]
iv = ["INCOM0", "RANK", "SATFIN_R", "FINPROS_R"]

raw_control = ["AGE", "SEX", "EDUC", "ABLITINF", "TECHHARM"]
control = ["AGE", "SEX", "EDUC", "ABLITINF_R", "TECHHARM_R"]

weight = "FINALWT" # wls 

analysis_vars_raw = [raw_dv] + raw_iv + raw_control + [weight]

missing = [v for v in analysis_vars_raw if v not in df_raw.columns]
if missing:
    raise ValueError(f"데이터에 없는 변수: {missing}")

display(df_raw[analysis_vars_raw].describe().T)

## 3. 데이터 전처리

- `-8`은 공통 결측 코드로 처리한다.
- `ABLITINF`의 `-1`은 비해당으로 보고 결측 처리한다.
- `GIVEPIFR`, `SATFIN`, `FINPROS`, `ABLITINF`, `TECHHARM`은 분석 방향에 맞게 역코딩한다.
- `INCOM0`는 `log_INCOM0 = log(INCOM0 + 1)`로 변환한다.

In [ ]:
df = df_raw[analysis_vars_raw].copy()

# 공통 결측 코드 처리
for col in analysis_vars_raw:
    df[col] = df[col].replace(-8, np.nan)

# ABLITINF의 -1은 비해당으로 보고 결측 처리
df["ABLITINF"] = df["ABLITINF"].replace(-1, np.nan)

# 종속변수 개인정보 제공 태도 역코딩: 원척도 1=매우 동의, 5=매우 반대
# 분석척도는 1=매우 반대, 5=매우 동의
df["GIVEPIFR_R"] = np.where(df["GIVEPIFR"].isin([1, 2, 3, 4, 5]), 6 - df["GIVEPIFR"], np.nan)

# 독립변수/통제변수 역코딩
# SATFIN_R: 1=매우 불만족, 5=매우 만족
# FINPROS_R: 1=상당히 나빠질 것, 5=상당히 좋아질 것
# ABLITINF_R: 1=매우 낮음, 5=매우 높음
for v in ["SATFIN", "FINPROS", "ABLITINF"]:
    df[f"{v}_R"] = np.where(df[v].isin([1, 2, 3, 4, 5]), 6 - df[v], np.nan)

# TECHHARM_R: 값이 높을수록 기술위험 인식이 높도록 역코딩
df["TECHHARM_R"] = np.where(df["TECHHARM"].isin([1, 2, 3, 4, 5]), 6 - df["TECHHARM"], np.nan)

# INCOM0 변수 로그 변환
df["log_INCOM0"] = np.log1p(df["INCOM0"].where(df["INCOM0"] >= 0, np.nan))


display(df.head())
display(df[["GIVEPIFR", "GIVEPIFR_R", "SATFIN_R", "FINPROS_R", "ABLITINF_R", "log_INCOM0", "TECHHARM_R"]].head())
display(df.isna().sum())

In [ ]:
# NaN 값 제거

analysis_vars = [
    "GIVEPIFR_R",
    "log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R",
    "AGE", "SEX", "EDUC", "ABLITINF_R", "TECHHARM_R",
    "FINALWT",
]

adt = df.dropna(subset=analysis_vars).copy().reset_index(drop=True)

print(f"결측 제거 전 df 행 수: {len(df)}")
print(f"결측 제거 후 adt 행 수: {len(adt)}")
print(f"제거된 행 수: {len(df) - len(adt)}")

display(adt.head())
display(adt[analysis_vars].isna().sum())

## 4. 기술통계와 상관관계

In [ ]:
summary_vars = [
    "GIVEPIFR", "GIVEPIFR_R",
    "INCOM0", "log_INCOM0", "RANK",
    "SATFIN", "SATFIN_R", "FINPROS", "FINPROS_R",
    "AGE", "SEX", "EDUC", "ABLITINF", "ABLITINF_R", "TECHHARM", "TECHHARM_R",
]
display(adt[summary_vars].describe().T)

corr_vars = ["GIVEPIFR_R", "INCOM0", "log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R", "ABLITINF_R", "TECHHARM_R"]
corr_matrix = adt[corr_vars].corr(method="pearson")
display(corr_matrix)

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson cor"},
)
plt.title("Pearson correlation")
plt.tight_layout()
plt.show()

## 5. 등분산성 검정

회귀 가정 점검용 Breusch-Pagan 검정과 White 검정.

In [ ]:
assumption_formula_main = (
    "GIVEPIFR_R ~ log_INCOM0 + RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + EDUC + ABLITINF_R + TECHHARM_R"
)

# wresid는 statsmodels가 WLS 계산에 사용한 가중 잔차다.
# BP/White 검정 함수는 상수항이 있는 설명변수 행렬을 요구하므로 exog는 원 설계행렬을 사용한다.
assumption_wls = smf.wls(
    formula=assumption_formula_main,
    data=adt,
    weights=adt["FINALWT"]
).fit()

bp = het_breuschpagan(assumption_wls.wresid, assumption_wls.model.exog)
white = het_white(assumption_wls.wresid, assumption_wls.model.exog)

heteroskedasticity_results = pd.DataFrame([{
    "model": "ASSUMPTION_WLS_GIVEPIFR_R_logINCOM0_main",
    "nobs": int(assumption_wls.nobs),
    "bp_lm_stat": bp[0],
    "bp_lm_pvalue": bp[1],
    "bp_f_stat": bp[2],
    "bp_f_pvalue": bp[3],
    "white_lm_stat": white[0],
    "white_lm_pvalue": white[1],
    "white_f_stat": white[2],
    "white_f_pvalue": white[3],
    "bp_heteroskedasticity_5pct": bp[1] < 0.05,
    "white_heteroskedasticity_5pct": white[1] < 0.05,
}])

display(heteroskedasticity_results)


질문: BP/White 모두 5% 수준에서 이분산성이 있는 것으로 나온다. 강건표준오차를 사용해야 할까? (회귀계수는 그대로 두고, 표준오차와 p값만 이분산성에 덜 민감하게 계산)

혹은 
  - 주 분석: FINALWT를 반영한 WLS
  - 보조 확인: WLS + HC3 강건표준오차

  두 분석을 진행한 후 두 결과의 방향성과 유의성이 크게 달라지는지 비교?
  > 만약, HC3의 결과와 wls의 결과가 거의 같으면 결과의 신뢰성이 높아지며, 유의성이 크게 바뀐다면 wls의 p값 해석을 유의해야 한다. 

## 6. 주 분석: WLS 회귀분석

### 분석1
종속변수: `GIVEPIFR_R`(값이 높을수록 개인정보 제공 동의 수준이 높다)  
독립변수: `log_INCOM0 , RANK, SATFIN_R, FINPROS_R`  
통제변수: `AGE`, `SEX`, `EDUC`

In [ ]:
formula_basic_wls = (
    "GIVEPIFR_R ~ log_INCOM0 + RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + EDUC"
)

basic_wls_model = smf.wls(
    formula=formula_basic_wls,
    data=adt,
    weights=adt["FINALWT"]
).fit()

basic_key_terms = ["log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R", "AGE", "C(SEX)[T.2.0]", "EDUC"]

basic_wls_result_table = pd.DataFrame({
    "term": basic_wls_model.params.index,
    "coef": basic_wls_model.params.values,
    "std_err": basic_wls_model.bse.values,
    "p_value": basic_wls_model.pvalues.values,
    "ci_low": basic_wls_model.conf_int().iloc[:, 0].values,
    "ci_high": basic_wls_model.conf_int().iloc[:, 1].values,
})

# 핵심 변수 결과표
display(basic_wls_result_table[basic_wls_result_table["term"].isin(basic_key_terms)].sort_values("term"))

# 전체 회귀 결과표
print("=" * 100)
print("BASIC_WLS_GIVEPIFR_R_logINCOM0")
print(basic_wls_model.summary())

### 분석2
종속변수: `GIVEPIFR_R`(값이 높을수록 개인정보 제공 동의 수준이 높다)  
독립변수: `log_INCOM0 , RANK, SATFIN_R, FINPROS_R`  
통제변수: `AGE`, `SEX`, `EDUC`, `ABLITINF_R`, `TECHHARM_R`

In [ ]:
main_formula = (
    "GIVEPIFR_R ~ log_INCOM0 + RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + EDUC + ABLITINF_R + TECHHARM_R"
)

wls_model = smf.wls(
    formula=main_formula,
    data=adt,
    weights=adt["FINALWT"]
).fit()

wls_result_table = pd.DataFrame({
    "term": wls_model.params.index,
    "coef": wls_model.params.values,
    "std_err": wls_model.bse.values,
    "p_value": wls_model.pvalues.values,
    "ci_low": wls_model.conf_int().iloc[:, 0].values,
    "ci_high": wls_model.conf_int().iloc[:, 1].values,
})

# 핵심 변수 결과표
key_terms = ["log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R", "ABLITINF_R", "TECHHARM_R"]
display(wls_result_table[wls_result_table["term"].isin(key_terms)].sort_values("term"))

# 전체 회귀 결과표
print("=" * 100)
print("MAIN_WLS_GIVEPIFR_R_logINCOM0")
print(wls_model.summary())

## 8. 다중공선성 점검: VIF

In [ ]:
vif_formula_main = (
    "GIVEPIFR_R ~ log_INCOM0 + RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + EDUC + ABLITINF_R + TECHHARM_R"
)


def calculate_vif(formula, data, model_name):
    y, X = dmatrices(formula, data=data, return_type="dataframe")
    rows = []
    for i, col in enumerate(X.columns):
        if col == "Intercept":
            continue
        rows.append({
            "model": model_name,
            "term": col,
            "vif": variance_inflation_factor(X.values, i),
        })
    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)

vif_results = calculate_vif(vif_formula_main, adt, "VIF_main_GIVEPIFR_logINCOM0_model")
display(vif_results)

## 해석 기준 요약

- `GIVEPIFR_R`은 값이 높을수록 할인/무료상품을 위한 개인정보 제공 동의가 높다는 의미이다.
- WLS 회귀에서 계수가 양수이면 해당 변수가 증가할수록 개인정보 제공 동의 수준이 높아지는 방향이다.
- WLS 회귀에서 계수가 음수이면 해당 변수가 증가할수록 개인정보 제공 동의 수준이 낮아지는 방향이다.
- `SATFIN_R`, `FINPROS_R`, `ABLITINF_R`는 모두 값이 높을수록 각각 가계상태 만족, 가계경제 전망 긍정, 정보판별능력이 높다는 뜻이다.
- `TECHHARM_R`는 값이 높을수록 기술위험 인식이 높다는 뜻이다.
